# 05 -- Animation (DCC process D1, dissemination/outreach)

**SEA-FORWARD** OceanPrediction-A toolkit

Time-evolving companion to `01_visualisation.ipynb`'s static maps and
`03_exercises.ipynb`'s single-frame eddy-detection exercise: four kinds of
animation over a CROCO run, built on `sftools.animate`.

| Section | Animation | Function |
|---|---|---|
| 1 | SSH with detected eddies | `animate_ssh_eddies` |
| 2 | Current vectors over speed | `animate_currents` |
| 3 | Scalar field (temperature/salinity) | `animate_scalar` |
| 4 | Lagrangian particle advection | `advect_particles` |

**Dependency note.** `sftools.animate` uses **py-eddy-tracker** for eddy
identification (Section 1) and particle advection (Section 4) -- a real,
hard dependency (the module raises `ImportError` on import if it's
missing), already listed in `environment.yml`'s pip dependencies. If the
import below fails, install it with `pip install pyEddyTracker` inside the
`seaforward` conda environment and restart the kernel; everything else in
this toolkit works without it.

Every function here writes an animation file (`.mp4` via ffmpeg, or `.gif`
via Pillow) rather than returning a plot -- this notebook picks whichever
writer is actually available and tells you which, rather than assuming.

*Language note (FR-09):* markdown and docstrings are in English; French
translation is coordinated separately with the documentation team.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from matplotlib.animation import writers as _mpl_writers
from IPython.display import Video, Image, display

import sftools.postprocess as pp

try:
    import sftools.animate as anim
except ImportError as e:
    raise ImportError(
        "sftools.animate requires py-eddy-tracker, which could not be imported. "
        "Install it with: pip install pyEddyTracker  (inside the seaforward conda "
        "environment), then restart this notebook's kernel. Original error: "
        f"{e}"
    ) from e

import _demo_data

CROCO_HIS, REFERENCE, YORIG, IS_DEMO = _demo_data.get_paths()
if IS_DEMO:
    print("!! DEMO DATA !! see notebooks/_demo_data.py -- these animations show a")
    print("   small synthetic stand-in, not a real CROCO run.")

WRITER = "ffmpeg" if _mpl_writers.is_available("ffmpeg") else (
    "pillow" if _mpl_writers.is_available("pillow") else None)
EXT = ".mp4" if WRITER == "ffmpeg" else ".gif"
if WRITER is None:
    raise RuntimeError(
        "Neither the ffmpeg nor the pillow matplotlib animation writer is "
        "available. Install one of them (e.g. `conda install ffmpeg` or "
        "`pip install pillow`) before running this notebook.")
print(f"using animation writer: {WRITER}  (output extension: {EXT})")

OUT_DIR = "./_animation_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


def show(path):
    '''Display an animation file inline (Video for mp4, Image for gif).'''
    if path.endswith(".gif"):
        display(Image(filename=path))
    else:
        display(Video(path, embed=True))


## 1. SSH with detected eddies

`animate_ssh_eddies` regrids CROCO's SSH (and surface currents) onto a
regular lon/lat grid each frame, applies py-eddy-tracker's
`bessel_high_filter` (a spatial high-pass, isolating the mesoscale
SSH-anomaly signal), then calls its `eddy_identification` (closed SSH
contours with amplitude and shape-error acceptance criteria -- see
`sftools/animate.py`'s docstring for exactly which py-eddy-tracker calls
this wraps).

`filter_km` is tuned down from the function's 500 km default for the small
demo domain here (~2 degrees across) -- on a domain that size, a 500 km
high-pass would filter out virtually everything. Widen it back out for a
full-size regional run.


In [ ]:
ssh_eddies_path = os.path.join(OUT_DIR, f"ssh_eddies{EXT}")
anim.animate_ssh_eddies(CROCO_HIS, out=ssh_eddies_path, Yorig=YORIG,
                        filter_km=50 if IS_DEMO else 500, fps=4)
show(ssh_eddies_path)


## 2. Current vectors over speed

`animate_currents` draws a quiver of current vectors over a speed-shaded
background, at any fixed depth (`depth_m=None` for the surface).


In [ ]:
currents_path = os.path.join(OUT_DIR, f"currents_surface{EXT}")
anim.animate_currents(CROCO_HIS, out=currents_path, depth_m=None, Yorig=YORIG,
                      skip=3, fps=4)
show(currents_path)


In [ ]:
# same, but at 100 m depth -- compare against the surface animation above to
# see how the near-surface jet (Exercise 3 of 03_exercises.ipynb) weakens/
# shifts with depth
currents_100m_path = os.path.join(OUT_DIR, f"currents_100m{EXT}")
anim.animate_currents(CROCO_HIS, out=currents_100m_path, depth_m=100, Yorig=YORIG,
                      skip=3, fps=4)
show(currents_100m_path)


## 3. Scalar field animation (temperature, salinity)

`animate_scalar` animates any CROCO scalar field at any depth -- a plain
time-evolving pcolormesh, no eddy tracking or particle advection involved.


In [ ]:
sst_path = os.path.join(OUT_DIR, f"sst{EXT}")
anim.animate_scalar(CROCO_HIS, var="temp", out=sst_path, depth_m=None,
                    Yorig=YORIG, fps=4)
show(sst_path)


In [ ]:
salt_path = os.path.join(OUT_DIR, f"salinity_surface{EXT}")
anim.animate_scalar(CROCO_HIS, var="salt", out=salt_path, depth_m=None,
                    Yorig=YORIG, fps=4)
show(salt_path)


## 4. Lagrangian particle advection

`advect_particles` seeds a patch of virtual particles and advects them
through the CROCO current field using py-eddy-tracker's own advection
routine (the same workflow as their `pet_advect.py` example) -- a visceral
way to see transport and stirring that a static vector map can't show.

Seeded at the same coastal reference point used throughout
`03_exercises.ipynb`, so you can compare where water actually goes against
the coastal-jet section computed there.


In [ ]:
ds = pp.open_history(CROCO_HIS, Yorig=YORIG)
clon, clat, cmask = pp.lonlatmask(ds)
lon0 = float(clon[cmask > 0][0])
lat0 = float(clat[cmask > 0][0])
n_time = ds.sizes["time"]
ds.close()
print(f"seeding particles at ({lon0:.2f}, {lat0:.2f})")

particles_path = os.path.join(OUT_DIR, f"particles{EXT}")
anim.advect_particles(CROCO_HIS, out=particles_path, lon0=lon0, lat0=lat0,
                      half_width_deg=0.15 if IS_DEMO else 1.0,
                      n_particles=300 if IS_DEMO else 2000,
                      days=max(n_time - 1, 1), dt_hours=24 if IS_DEMO else 3,
                      Yorig=YORIG, fps=4)
show(particles_path)


---
## Notes

- **DCC linkage (FR-12):** animation is a D1 (downstream/dissemination)
  product -- it consumes validated C1 output (ideally already checked by
  `02_validation.ipynb`) and produces outreach/diagnostic material, not a
  new analysis result in its own right.
- **Outputs** land in `notebooks/_animation_outputs/` (created above) --
  not committed to the repository; regenerate as needed. Delete that
  folder to force a clean re-render.
- **Runtime:** eddy identification and particle advection both regrid the
  curvilinear CROCO grid to a regular one *every frame* -- expect this
  notebook to take noticeably longer than `02_validation.ipynb` on a full-
  size regional run with many time steps. `stride`/`tstart`/`tend` (Sections
  1-3) and `dt_hours` (Section 4) all trade detail for runtime.
- **QA:** designed to execute without errors from a fresh kernel restart,
  using `_demo_data.py`'s fallback if real CROCO output isn't present --
  same convention as the other notebooks. The one exception QA can't paper
  over is the py-eddy-tracker dependency itself (see the setup cell); if
  your environment is missing it, install it and restart the kernel.
